In [81]:
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

from autobound.causalProblem import causalProblem
from autobound.DAG import DAG
from autobound.Query import Query

import io

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


$\mathcal{G}:$
```
          [R₃]         [R₁]     [R₂]
        ↙    ↘       ↙    ↘  ↙    ↘
       A ───→ B ───→ X ────→ M ────→ Y 
              ↑       │      ↑
              │       ↓      │
             [R₄]────→C─────→D
```

We have $\theta=(Y=1|do(X=1,M=1))$.

The mission of this notebook:
Calculate 
1. $a_1$: $pot(Y,M|do(X=1))$
2. $a_2$: $pot(Y,M|do(X=0))$


## Step 0: Simulate SCM

Remark: W.l.o.g. We can assume a simpler graph
$X \to M \to Y$ (@TODO Change that! Assume the actual graph)

In [45]:
N_samples = 1000000
np.random.seed(42)

# We are simulating a ground truth SCM. That is, we will 
# 1. randomly pick all root variables, 
# 2. and sample others as some arbitrary boolean function of their parents.
# We will simulate the actual confounders (not response types!) and we will make them binary. (That keeps the structural equations boolean)

# 1. Simulate Root variables (confounders, A)
A  = np.random.binomial(1, 0.3, N_samples)
U3 = np.random.binomial(1, 0.7, N_samples)
U4 = np.random.binomial(1, 0.6, N_samples)
U1 = np.random.binomial(1, 0.1, N_samples)
U2 = np.random.binomial(1, 0.8, N_samples)

# 2. Structural equations (B, X, C, D, M, Y)
B = A ^ U3
X = B ^ U1
C = X ^ U4
D = C ^ 1  # D = NOT(C)
M = (U1 & U2) ^ (X | D)
Y = U2 & M

ground_truth_data = pd.DataFrame({
    'A': A,
    'B': B,
    'X': X,
    'C': C,
    'D': D,
    'M': M,
    'Y': Y,
    'U1': U1,
    'U2': U2,
    'U3': U3,
    'U4': U4
})

## Step 1: Compute $\mathcal{W}_\emptyset$

### If we use the full DAG, the problem becomes computationally infeasible.

In [47]:
# Observational data
obs = ground_truth_data.drop(columns=['U1', 'U2', 'U3', 'U4'])
# compute probabilities for every combination of values (000, ... 111) and safe them as csv: A,B,X...,prob
obs_counts = obs.value_counts().reset_index(name='count')

# Round probabilities and force exact normalization after rounding
obs_counts['prob'] = (obs_counts['count'] / N_samples).round(6)
diff = round(1.0 - obs_counts['prob'].sum(), 6)

if diff != 0:
    idx = obs_counts['count'].idxmax()  # adjust the largest-mass row
    obs_counts.loc[idx, 'prob'] = round(obs_counts.loc[idx, 'prob'] + diff, 6)

obs_counts.drop(columns=['count']).to_csv('data/figure-4-obs.csv', index=False)

In [188]:
obs_counts.drop(columns=['count'])

,X,M,Y,prob
0,1,1,1,0.417475
1,0,0,0,0.251882
2,0,1,1,0.149284
3,1,1,0,0.113045
4,0,1,0,0.034809
5,1,0,0,0.033505


In [49]:
dag = DAG()
dag.from_structure("A->B, B->X, X->C, C->D, D->M, X->M, M->Y, " \
"U3->A, U3->B, U4->B, U4->C, U1->X, U1->M, U2->M, U2->Y", unob='U1,U2,U3,U4')
problem = causalProblem(dag)
problem.load_data('data/figure-4-obs.csv')
problem.add_prob_constraints()
problem.set_estimand(problem.query('Y(X=1, M=1)=1'))
program = problem.write_program()
program.to_pip('figure4.pip')

(lb, ub) = program.run_pyomo('ipopt', verbose=True, parallel=True)   # Converges to a locally infeasible point.
#(lb, ub) = program.run_pyomo('couenne', verbose=True, parallel=True)   # Runs forever
# (lb, ub) = program.run_couenne( verbose=True, epsilon=0.9, theta=0.9)   # Runs forever.
calW_empty = ub-lb

Ipopt 3.12.12: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

This is Ipopt version 3.12.12, running with linear solver mumps.
NOTE: Other linear solvers might be more efficient (see Ipopt documentation).

Number of nonzeros in equality constraint Jacobian...:     1561
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:     4492

Total number of variables............................:      293
                     variables with only lower bounds:        0
                variables with lower and upper bounds:      293
                     variables with only upper bounds:        0
Tot

### That is why we set up the reduced program!
Recall that $R_\theta = \{R_2\}$ and therefore
* $D^*(R_\theta)= \{R_1, R_2, X, M, Y\}$.
* $D_\theta^* = \{\{R_1, R_2, X, M, Y\}\}$
* $R_\theta^* = \{R_1, R_2\}$

As shown in the paper, it suffices to optimize over $R_*^\theta$. Meaning we only have to model $D^*(R_\theta)$.

In [63]:
reduced_data = ground_truth_data.drop(columns=['U3', 'A', 'B', 'U4', 'C', 'D'])
obs = reduced_data.drop(columns=['U1', 'U2'])
obs_counts = obs.value_counts().reset_index(name='count')

# Round probabilities and force exact normalization after rounding
obs_counts['prob'] = (obs_counts['count'] / N_samples).round(40)

#make sure it's a valid distribution
if diff != 0:
    print(f"Adjusting for rounding error of {diff} by modifying the largest-mass row.")
    idx = obs_counts['count'].idxmax()  # adjust the largest-mass row
    obs_counts.loc[idx, 'prob'] = round(obs_counts.loc[idx, 'prob'] + diff, 6)

obs_counts.drop(columns=['count']).to_csv('data/figure-4-obs.csv', index=False)

In [172]:
dag = DAG()
dag.from_structure("X->M, M->Y, U1->X, U1->M, U2->M, U2->Y, X->Y", unob='U1,U2')
problem = causalProblem(dag)
problem.load_data('data/figure-4-obs.csv')
problem.add_prob_constraints()
problem.set_estimand(problem.query('Y(X=1, M=1)=1'))
program = problem.write_program()
program.to_pip('figure4.pip')

(lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True) 
print(f"P(θ)∈[{lb:.2f}, {ub:.2f}]")
calW_empty = ub - lb
print(f"calW_empty = {calW_empty:.8f}")

P(θ)∈[0.42, 0.89]
calW_empty = 0.46948000


## Step 2: Compute $\mathcal{W_{a}}$
Summarizing my findings.
* $pot(Y,M |do(X=0))=0$
* $pot(Y,M |do(X=1))>0$


### Illustration: True $W_a$
Below is a code that modifies the structural equations and compute the distrubtion $P(Y,M|do(x))$ we would if we actually were to perform an intervention on $X$. This is purely for illustratiative purposes.

In [194]:
# Add a single experiment P(Y=1, M=1 | do(X=1)) first.

def getIntervention(doX_val):
    M_doX = M = (U1 & U2) ^ (doX_val | D) #Set X=1 in structural equation
    Y_doX = U2 & M_doX #Note: If M_doX=0, Y_doX=0 always. So P(X=0,Y=1)=0
    doX = pd.DataFrame({
        'M_doX': M_doX,
        'Y_doX': Y_doX,
    })

    doX_counts = doX.value_counts().reset_index(name='count')

    # Round probabilities and force exact normalization after rounding
    doX_counts['prob'] = (doX_counts['count'] / N_samples).round(40)

    return doX_counts.drop(columns=['count'])
print(getIntervention(1))

   M_doX  Y_doX      prob
0      1      1  0.719872
1      1      0  0.199767
2      0      0  0.080361


### Algorithm: Approximating W_a(p_a) via Grid Search

We want to compute the worst-case width W_a = max_{p_a} W(p_a) of the identified set
for θ = P(Y=1 | do(X=1, M=1)), maximized over all possible experimental outcomes p_a
of experiment a = (Y, M | do(X=x)).

Since we don't know the true interventional distribution p_a, we search over all
candidate probability vectors p = (p_00, p_01, p_10, p_11) that could plausibly arise
from the experiment do(X=x).

1. **Bound the search space** using observational data (Tian & Pearl, 2000, eq. 22):
   For each outcome (M=m, Y=y), the interventional probability is bounded by:
     - Lower bound: P(X=x, M=m, Y=y)
     - Upper bound: 1 - P(X=x) + P(X=x, M=m, Y=y)
   These bounds follow from consistency (the interventional and observational
   distributions must agree when X already takes value x) and non-negativity
   of probabilities.

2. **Discretize** each bounded interval into a grid with step size 1/n
   (controlled by the `granularity` parameter n).

3. **Enumerate** all grid vectors (p_00, p_01, p_10, p_11) that:
   - Sum to 1 (valid probability distribution)
   - Fall within the bounds from step 1

4. **For each candidate vector**, set up the partial identification problem:
   - Load observational data as constraints
   - Add the candidate interventional distribution as additional constraints
   - Optimize (min and max) the estimand θ using the Couenne global solver
   - Record the width (upper bound − lower bound)

5. **Return the maximum width** across all feasible candidate vectors.
   This is the worst-case W_a: the largest the identified set could be,
   no matter what the experiment reveals. The potency of the experiment is then
   pot(a) = W_∅ − W_a.


In [189]:
def W_X(p_config, X_val):
    problem = causalProblem(dag)
    problem.load_data('data/figure-4-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1, M=1)=1'))

    for _, row in p_config.iterrows():
        m, y = int(row.M_doX), int(row.Y_doX)
        lhs = problem.query(f'M(X={X_val})={m}&Y(X={X_val})={y}')
        problem.add_constraint(lhs - Query(row.prob))
    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb


def calW_X(X_val, granularity):
    n = granularity
    obs = obs_counts.drop(columns=['count'])

    # Bounds from Tian & Pearl (2000) eq. 22
    p_x = obs[obs['X'] == X_val]['prob'].sum()

    def get_bounds(m, y):
        mask = (obs['X'] == X_val) & (obs['M'] == m) & (obs['Y'] == y)
        p_joint = obs.loc[mask, 'prob'].sum()
        return (p_joint, 1 - p_x + p_joint)

    lb_00, ub_00 = get_bounds(0, 0)
    lb_01, ub_01 = get_bounds(0, 1)
    lb_10, ub_10 = get_bounds(1, 0)
    lb_11, ub_11 = get_bounds(1, 1)

    print(f"Bounds for do(X={X_val}):")
    print(f"  p_00 ∈ [{lb_00:.4f}, {ub_00:.4f}]")
    print(f"  p_01 ∈ [{lb_01:.4f}, {ub_01:.4f}]")
    print(f"  p_10 ∈ [{lb_10:.4f}, {ub_10:.4f}]")
    print(f"  p_11 ∈ [{lb_11:.4f}, {ub_11:.4f}]")

    W_approx = []

    a_lo, a_hi = int(np.ceil(lb_00 * n)), int(np.floor(ub_00 * n))
    b_lo, b_hi = int(np.ceil(lb_01 * n)), int(np.floor(ub_01 * n))
    c_lo, c_hi = int(np.ceil(lb_10 * n)), int(np.floor(ub_10 * n))

    for a in range(a_lo, a_hi + 1):
        for b in range(b_lo, min(b_hi, n - a) + 1):
            for c in range(c_lo, min(c_hi, n - a - b) + 1):
                d = n - a - b - c
                p_11 = d / n
                if p_11 < lb_11 - 1e-9 or p_11 > ub_11 + 1e-9:
                    continue

                p_00, p_01, p_10 = a / n, b / n, c / n

                doX_hyp = pd.DataFrame({
                    'M_doX': [1, 1, 0, 0],
                    'Y_doX': [1, 0, 0, 1],
                    'prob':  [p_11, p_10, p_00, p_01]
                })
                try:
                    W_approx.append(W_X(doX_hyp, X_val))
                except Exception as e:
                    print(f"Skipping p=({p_00:.2f}, {p_01:.2f}, {p_10:.2f}, {p_11:.2f}): {e}")
                    continue
    return max(W_approx)


In [192]:
calW_1 = calW_X(1, 16)
calW_0 = calW_X(0, 16)

Bounds for do(X=1):
  p_00 ∈ [0.0335, 0.4695]
  p_01 ∈ [0.0000, 0.4360]
  p_10 ∈ [0.1130, 0.5490]
  p_11 ∈ [0.4175, 0.8535]
Bounds for do(X=0):
  p_00 ∈ [0.2519, 0.8159]
  p_01 ∈ [0.0000, 0.5640]
  p_10 ∈ [0.0348, 0.5988]
  p_11 ∈ [0.1493, 0.7133]


In [193]:
print(f"W_empty = {calW_empty:.8f}")
print(f"W_0 = {calW_0:.8f}, W_1 = {calW_1:.8f}")
print(f"pot(Y,M|do(X=1))={calW_empty - calW_1:.8f}, pot(Y,M|do(X=0))={calW_empty - calW_0:.8f}")

W_empty = 0.46948000
W_0 = 0.46948057, W_1 = 0.43750032
pot(Y,M|do(X=1))=0.03197968, pot(Y,M|do(X=0))=-0.00000057
